# Práctica 4. Detección de características (SIFT)

En esta práctica hemos hecho una aplicación con ventana propia (Tkinter) que se abre fuera del notebook y resuelve los apartados a, b y c del ejercicio 1. Decidimos hacerla en una ventana aparte y no con controles dentro del notebook porque así se puede trabajar con las imágenes a tamaño completo y seleccionar el área de interés directamente con el ratón sobre la imagen.

| Apartado | Qué resolvemos | Pestaña de la ventana |
|---|---|---|
| **a** | Cambiar los parámetros del detector SIFT desde la interfaz | 1 · Parámetros SIFT |
| **b** | Elegir un área de interés en una imagen médica y en una telemétrica | 2 · Área de interés |
| **c** | Buscar esa área (recuadrada en rojo) en versiones trasladadas, escaladas y rotadas, ajustando los parámetros para mejorar la detección | 3 · Búsqueda |

Trabajamos con una resonancia magnética cerebral (`Medica.jpg`) y una imagen de satélite (`Telemetrica.jpg`), guardadas en la misma carpeta que el notebook. Si no encuentra alguna, la aplicación genera una imagen sintética para poder arrancar y lo avisa en pantalla.

Dependencias: `opencv-python` (4.4 o superior, por `cv2.SIFT_create`), `numpy` y `matplotlib`. Tkinter viene incluido con Python.

## 1. Librerías y paleta de colores

Usamos OpenCV para SIFT y el procesamiento de imágenes, NumPy para las operaciones con matrices y Tkinter para la ventana. De Matplotlib solo aprovechamos el motor de dibujo en segundo plano, para el histograma de tamaños y para el resumen visual, sin abrir ninguna figura aparte.

Para la interfaz elegimos una paleta de tonos lilas, que dejamos reunida en el diccionario `CL` para no repetir los colores por el código.

In [19]:
import os
import re
import json
import math
import time
import base64
import numpy as np
import cv2
import tkinter as tk
from tkinter import ttk, filedialog
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg

# Paleta lila de toda la interfaz
CL = dict(
    fondo='#F6F0FD', tarjeta='#EBDDFA', borde='#C9B1EC', suave='#D9C2F5',
    medio='#B794F4', fuerte='#7E57C2', oscuro='#4A2C7A', texto='#3B2563',
    punto=(190, 70, 255), match=(159, 122, 234), ok='#6B46C1', aviso='#B8286B')

## 2. Funciones de visión

Separamos toda la parte de visión de la interfaz para poder probarla por su cuenta. Las decisiones que tomamos:

- **Carga de imágenes con `imdecode`.** `cv2.imread` falla en Windows si la ruta tiene tildes, y nuestras carpetas las tienen. Además reducimos a 1600 px de lado las imágenes muy grandes para que SIFT no tarde demasiado al mover los parámetros.
- **Preprocesado opcional** (NOTA 2 del enunciado): suavizado gaussiano, CLAHE o ambos, aplicado igual al área de interés y a la imagen donde se busca.
- **SIFT con todos sus parámetros:** `nfeatures`, `nOctaveLayers`, `contrastThreshold`, `edgeThreshold` y `sigma`.
- **Emparejamiento** con KNN (k=2) y el ratio test de Lowe, como en el ejemplo de la práctica. Se puede usar fuerza bruta o FLANN.
- **Localización con RANSAC.** Permitimos dos modelos: una *similitud* (traslación + escala + rotación), que es justo lo que cambiamos en el apartado c, y una *homografía*, más general y pensada para el apartado d.
- **Validación del resultado**, para cumplir la NOTA 1 ("si no encuentra el área, debería notificarlo"). RANSAC siempre devuelve alguna transformación si tiene suficientes puntos, así que no basta con comprobar que existe. Exigimos un mínimo de inliers y que el cuadrilátero resultante sea convexo y de un tamaño razonable. Si algo falla, la función devuelve el motivo en texto.
- **Métricas.** Cuando conocemos la transformación aplicada, calculamos el IoU entre el cuadrilátero encontrado y el real, y el error medio de las cuatro esquinas.

In [20]:
MAX_LADO = 1600   # las imágenes más grandes se reducen para que SIFT vaya fluido


def leer_imagen(ruta):
    """Lee una imagen como RGB (funciona con tildes en la ruta) y la reduce si es enorme."""
    datos = np.fromfile(ruta, dtype=np.uint8)
    bgr = cv2.imdecode(datos, cv2.IMREAD_COLOR)
    if bgr is None:
        raise ValueError(f'No se pudo decodificar «{ruta}»')
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    h, w = rgb.shape[:2]
    if max(h, w) > MAX_LADO:
        f = MAX_LADO / max(h, w)
        rgb = cv2.resize(rgb, (round(w * f), round(h * f)), interpolation=cv2.INTER_AREA)
    return rgb


def preprocesar(rgb, p):
    """Escala de grises + suavizado y/o CLAHE según la opción elegida."""
    gris = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    modo = p['preproc']
    if 'Suavizado' in modo:
        k = p['ksize'] | 1
        gris = cv2.GaussianBlur(gris, (k, k), 0)
    if 'CLAHE' in modo:
        gris = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gris)
    return gris


def crear_sift(p):
    return cv2.SIFT_create(nfeatures=int(p['nfeatures']),
                           nOctaveLayers=int(p['nOctaveLayers']),
                           contrastThreshold=float(p['contrastThreshold']),
                           edgeThreshold=float(p['edgeThreshold']),
                           sigma=float(p['sigma']))


def extraer(rgb, p):
    """Puntos clave y descriptores SIFT de una imagen RGB."""
    kp, des = crear_sift(p).detectAndCompute(preprocesar(rgb, p), None)
    return list(kp), des


def claves_sift(p):
    """Tupla con los parámetros que influyen en la extracción (sirve de clave de caché)."""
    return (p['nfeatures'], p['nOctaveLayers'], round(p['contrastThreshold'], 4),
            round(p['edgeThreshold'], 3), round(p['sigma'], 3), p['preproc'], p['ksize'])


def localizar(kp_t, des_t, kp_e, des_e, forma_t, p):
    """Busca la plantilla (área de interés) en la escena.

    Devuelve un diccionario con el resultado y, si no se encuentra, el motivo.
    """
    h, w = forma_t
    res = dict(encontrado=False, motivo='', n_kp_t=len(kp_t), n_kp_e=len(kp_e),
               n_buenos=0, n_inliers=0, H=None, quad=None, buenos=[], mascara=None)

    if des_t is None or len(kp_t) < 2:
        res['motivo'] = ('El área de interés apenas tiene textura y SIFT no encuentra puntos clave en ella. '
                         'Baja el umbral de contraste, prueba CLAHE o elige otra zona.')
        return res
    if des_e is None or len(kp_e) < 2:
        res['motivo'] = ('SIFT no detecta puntos clave en la imagen donde se busca. '
                         'Baja el umbral de contraste o aplica CLAHE.')
        return res

    if p['matcher'].startswith('FLANN'):
        matcher = cv2.FlannBasedMatcher(dict(algorithm=1, trees=5), dict(checks=64))
    else:
        matcher = cv2.BFMatcher(cv2.NORM_L2)
    pares = matcher.knnMatch(des_t, des_e, k=2)
    buenos = [pr[0] for pr in pares if len(pr) == 2 and pr[0].distance < p['ratio'] * pr[1].distance]
    res['buenos'] = buenos
    res['n_buenos'] = len(buenos)

    es_homografia = p['modelo'].startswith('Homograf')
    minimo = 4 if es_homografia else 3
    if len(buenos) < minimo:
        res['motivo'] = (f'Solo hay {len(buenos)} emparejamientos que pasan el ratio test (se necesitan al menos {minimo}). '
                         'Sube el umbral del ratio, baja el umbral de contraste o prueba otro preprocesado.')
        return res

    src = np.float32([kp_t[m.queryIdx].pt for m in buenos]).reshape(-1, 1, 2)
    dst = np.float32([kp_e[m.trainIdx].pt for m in buenos]).reshape(-1, 1, 2)
    if es_homografia:
        H, mask = cv2.findHomography(src, dst, cv2.RANSAC, float(p['ransac_thr']))
    else:
        A, mask = cv2.estimateAffinePartial2D(src, dst, method=cv2.RANSAC,
                                              ransacReprojThreshold=float(p['ransac_thr']))
        H = None if A is None else np.vstack([A, [0, 0, 1]])

    if H is None or mask is None:
        res['motivo'] = 'RANSAC no pudo ajustar ninguna transformación con los emparejamientos disponibles.'
        return res

    res['mascara'] = mask
    res['n_inliers'] = int(mask.sum())
    if res['n_inliers'] < p['min_inliers']:
        res['motivo'] = (f"Solo {res['n_inliers']} inliers tras RANSAC (mínimo exigido: {p['min_inliers']}). "
                         'Los emparejamientos no son coherentes entre sí: el área probablemente no está en la imagen '
                         'o hay que ajustar los parámetros.')
        return res

    esquinas = np.float32([[0, 0], [w, 0], [w, h], [0, h]]).reshape(-1, 1, 2)
    quad = cv2.perspectiveTransform(esquinas, H).reshape(-1, 2)
    if not np.all(np.isfinite(quad)):
        res['motivo'] = 'La transformación estimada es degenerada.'
        return res
    area = cv2.contourArea(quad.astype(np.float32))
    razon = area / float(w * h)
    if not cv2.isContourConvex(np.int32(np.round(quad))) or razon < 0.01 or razon > 100:
        res['motivo'] = ('La transformación estimada da un recuadro deforme o de tamaño absurdo; '
                         'se descarta como falso positivo.')
        return res

    res.update(encontrado=True, H=H, quad=quad)
    return res


def dibujar_busqueda(escena, res, grosor=3, quad_real=None):
    """Escena con el área encontrada recuadrada en rojo."""
    out = escena.copy()
    if quad_real is not None:
        cv2.polylines(out, [np.int32(np.round(quad_real)).reshape(-1, 1, 2)], True,
                      CL['punto'], 1, cv2.LINE_AA)
    if res['encontrado']:
        cv2.polylines(out, [np.int32(np.round(res['quad'])).reshape(-1, 1, 2)], True,
                      (255, 0, 0), grosor, cv2.LINE_AA)
    return out


def dibujar_emparejamientos(plantilla, kp_t, escena, kp_e, res):
    mask = res['mascara'].ravel().tolist() if res['mascara'] is not None else None
    return cv2.drawMatches(plantilla, kp_t, escena, kp_e, res['buenos'], None,
                           matchColor=CL['match'], singlePointColor=(200, 200, 200),
                           matchesMask=mask, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)


def iou_cuadrilateros(q1, q2, forma):
    """IoU entre dos cuadriláteros rasterizados sobre la escena."""
    h, w = forma[:2]
    m1 = np.zeros((h, w), np.uint8)
    m2 = np.zeros((h, w), np.uint8)
    cv2.fillPoly(m1, [np.int32(np.round(q1))], 1)
    cv2.fillPoly(m2, [np.int32(np.round(q2))], 1)
    union = np.logical_or(m1, m2).sum()
    return float(np.logical_and(m1, m2).sum() / union) if union else 0.0

## 3. Versiones de la imagen

El enunciado permite generar las versiones alteradas con un editor o con prácticas anteriores. Preferimos generarlas dentro de la propia aplicación con una única transformación afín: primero escala y rotación respecto al centro de la imagen, y después la traslación. El lienzo de salida se amplía para que ninguna parte quede recortada.

Generarlas así tiene una ventaja: como conocemos la matriz de la transformación, sabemos exactamente dónde queda el área de interés y podemos medir el acierto con el IoU. Aun así, la aplicación también acepta archivos alterados con un editor, aunque en ese caso no hay posición real con la que comparar.

Definimos además una lista de versiones predefinidas (traslación, tres escalas, cuatro rotaciones y una combinación) y una función que genera imágenes sintéticas de demostración para que la aplicación arranque aunque falten los archivos.

In [21]:
def transformar(rgb, tx=0, ty=0, escala=1.0, angulo=0.0):
    """Aplica escala + rotación (antihoraria, en grados) + traslación.

    Devuelve la imagen transformada y la matriz 3x3 original -> nueva.
    La traslación (tx, ty) desplaza la imagen respecto al centro de un lienzo ampliado.
    """
    h, w = rgb.shape[:2]
    R = cv2.getRotationMatrix2D((w / 2, h / 2), angulo, escala)
    esq = np.float32([[0, 0], [w, 0], [w, h], [0, h]]).reshape(-1, 1, 2)
    pts = cv2.transform(esq, R).reshape(-1, 2)
    x0, y0 = pts.min(axis=0)
    x1, y1 = pts.max(axis=0)
    mx, my = abs(tx), abs(ty)
    W = int(min(math.ceil(x1 - x0 + 2 * mx), 5000))
    H = int(min(math.ceil(y1 - y0 + 2 * my), 5000))
    M = R.copy()
    M[0, 2] += -x0 + mx + tx
    M[1, 2] += -y0 + my + ty
    out = cv2.warpAffine(rgb, M, (W, H), flags=cv2.INTER_LINEAR,
                         borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
    return out, np.vstack([M, [0, 0, 1]])


VERSIONES = {
    'Sin cambios':            dict(tx=0,   ty=0,   escala=1.0, angulo=0),
    'Traslación (+80, +50)':  dict(tx=80,  ty=50,  escala=1.0, angulo=0),
    'Escala 0.5':             dict(tx=0,   ty=0,   escala=0.5, angulo=0),
    'Escala 1.5':             dict(tx=0,   ty=0,   escala=1.5, angulo=0),
    'Escala 2.0':             dict(tx=0,   ty=0,   escala=2.0, angulo=0),
    'Rotación 30°':           dict(tx=0,   ty=0,   escala=1.0, angulo=30),
    'Rotación -45°':          dict(tx=0,   ty=0,   escala=1.0, angulo=-45),
    'Rotación 90°':           dict(tx=0,   ty=0,   escala=1.0, angulo=90),
    'Rotación 180°':          dict(tx=0,   ty=0,   escala=1.0, angulo=180),
    'Combinada (esc 0.7, rot 25°, tras -60,40)': dict(tx=-60, ty=40, escala=0.7, angulo=25),
}


def imagen_demo(clave, semilla=7):
    """Imagen sintética de demostración (solo si no se encuentra el archivo real)."""
    rng = np.random.default_rng(semilla)

    def ruido(n_h, n_w, sigma):
        r = cv2.GaussianBlur(rng.normal(0, 1, (n_h, n_w)).astype(np.float32), (0, 0), sigma)
        return (r - r.min()) / (r.max() - r.min() + 1e-6)

    if clave == 'Médica':
        n = 512
        yy, xx = np.mgrid[:n, :n].astype(np.float32)
        r = np.hypot(xx - n / 2, (yy - n / 2) * 1.12)
        img = 200 * np.exp(-((r - 215) / 8) ** 2)
        tejido = 70 + 90 * ruido(n, n, 5) + 35 * np.sin(r / 6 + 4 * ruido(n, n, 12))
        img += np.where(r < 200, tejido, 0)
        for cx, cy, a, b in [(235, 250, 14, 50), (290, 250, 14, 50), (262, 215, 30, 9)]:
            img -= 120 * np.exp(-(((xx - cx) / a) ** 2 + ((yy - cy) / b) ** 2))
        g = np.clip(img + rng.normal(0, 4, img.shape), 0, 255).astype(np.uint8)
        return cv2.cvtColor(g, cv2.COLOR_GRAY2RGB)

    h, w = 600, 900
    verde = np.dstack([40 + 40 * ruido(h, w, 3), 90 + 70 * ruido(h, w, 4), 40 + 30 * ruido(h, w, 3)])
    marron = np.dstack([110 + 60 * ruido(h, w, 15), 90 + 40 * ruido(h, w, 15), 60 + 30 * ruido(h, w, 15)])
    mezcla = ruido(h, w, 40)[..., None]
    img = np.where(mezcla > 0.5, verde, marron).astype(np.uint8).copy()
    for _ in range(45):
        x, y = int(rng.integers(0, w - 60)), int(rng.integers(0, h - 60))
        bw, bh = int(rng.integers(18, 60)), int(rng.integers(18, 60))
        tono = int(rng.integers(120, 230))
        cv2.rectangle(img, (x, y), (x + bw, y + bh), (tono, tono, tono - 10), -1)
        cv2.rectangle(img, (x, y), (x + bw, y + bh), (50, 50, 60), 1)
    for _ in range(9):
        p1 = (int(rng.integers(0, w)), int(rng.integers(0, h)))
        p2 = (int(rng.integers(0, w)), int(rng.integers(0, h)))
        cv2.line(img, p1, p2, (200, 200, 190), int(rng.integers(2, 5)), cv2.LINE_AA)
    for _ in range(250):
        cv2.circle(img, (int(rng.integers(0, w)), int(rng.integers(0, h))), int(rng.integers(3, 8)),
                   (20, int(rng.integers(60, 110)), 25), -1)
    return img

## 4. La aplicación

La clase `AppSIFT` construye la ventana y conecta los controles con las funciones anteriores. Está organizada en tres pestañas, una por apartado:

1. **Parámetros SIFT (a).** Los cinco parámetros del detector más el preprocesado, con configuraciones rápidas (por defecto, más sensible para imágenes de bajo contraste, más estricta y otra pensada para imágenes con mucho detalle). Se dibujan los puntos clave sobre la imagen y se muestran su número, el tamaño mediano y un histograma de tamaños.
2. **Área de interés (b).** Se arrastra el ratón sobre la imagen para dibujar el rectángulo, que también se puede afinar con valores numéricos. Debajo se ve el área recortada con sus puntos SIFT, y se avisa si tiene muy pocos, porque entonces será difícil localizarla. El área de cada imagen se guarda en `rois.json` para no tener que volver a elegirla.
3. **Búsqueda (c).** Se elige la versión (una predefinida, una personalizada con los deslizadores de traslación, escala y rotación, o un archivo alterado) y se busca el área elegida. El resultado se dibuja con el recuadro rojo y debajo se ven los emparejamientos. Los parámetros de SIFT y del emparejamiento se pueden cambiar sin salir de la pestaña y la búsqueda se repite sola. Si el área no aparece, un mensaje explica el motivo.

Los controles de SIFT de las pestañas 1 y 3 comparten las mismas variables de Tkinter, así que cualquier cambio en una se refleja en la otra. Para el informe añadimos una tabla con el resultado de todas las versiones predefinidas y un resumen visual que las junta en una sola imagen. Ambos se guardan en las carpetas `resultados` y `versiones`.

In [22]:
PRESETS_SIFT = {
    'OpenCV por defecto':                       dict(nfeatures=0,    capas=3, contraste=0.04,  borde=10.0, sigma=1.6),
    'Más sensible (bajo contraste, p. ej. RM)': dict(nfeatures=0,    capas=3, contraste=0.015, borde=15.0, sigma=1.6),
    'Más estricto (menos puntos, más fiables)': dict(nfeatures=1500, capas=3, contraste=0.08,  borde=8.0,  sigma=1.6),
    'Mucho detalle (imagen telemétrica)':       dict(nfeatures=3000, capas=4, contraste=0.04,  borde=10.0, sigma=1.4),
}
MODELOS = ['Similitud (traslación + escala + rotación)', 'Homografía']


# ----------------------------------------------------------------------------
# utilidades gráficas
# ----------------------------------------------------------------------------
def mezclar(c1, c2, t):
    a = [int(c1[i:i + 2], 16) for i in (1, 3, 5)]
    b = [int(c2[i:i + 2], 16) for i in (1, 3, 5)]
    return '#%02x%02x%02x' % tuple(round(a[i] + (b[i] - a[i]) * t) for i in range(3))


def a_photo(rgb):
    """Convierte una imagen RGB de NumPy en un PhotoImage de Tk (vía PNG en memoria)."""
    ok, buf = cv2.imencode('.png', cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
    return tk.PhotoImage(data=base64.b64encode(buf.tobytes()))


def ajustar(rgb, ancho, alto, max_ampliar=4.0):
    h, w = rgb.shape[:2]
    f = min(ancho / w, alto / h, max_ampliar)
    nuevo = (max(1, int(w * f)), max(1, int(h * f)))
    interp = cv2.INTER_AREA if f < 1 else cv2.INTER_LINEAR
    return cv2.resize(rgb, nuevo, interpolation=interp), f


def figura_a_rgb(fig):
    FigureCanvasAgg(fig).draw()
    return np.asarray(fig.canvas.buffer_rgba())[..., :3].copy()


def histograma(tamanos):
    fig = Figure(figsize=(3.6, 1.9), dpi=100, facecolor=CL['fondo'])
    ax = fig.add_subplot(111)
    ax.set_facecolor(CL['tarjeta'])
    ax.hist(tamanos, bins=25, color=CL['medio'], edgecolor=CL['oscuro'])
    ax.set_xlabel('Tamaño del punto (px)', fontsize=8, color=CL['texto'])
    ax.set_ylabel('Nº de puntos', fontsize=8, color=CL['texto'])
    ax.tick_params(colors=CL['texto'], labelsize=7)
    fig.tight_layout()
    return figura_a_rgb(fig)


class Visor(tk.Canvas):
    """Canvas que muestra una imagen ajustada a su tamaño y recuerda la escala usada."""

    def __init__(self, master, vacio='', **kw):
        super().__init__(master, bg=CL['tarjeta'], highlightthickness=1,
                         highlightbackground=CL['borde'], **kw)
        self.rgb = None
        self._foto = None
        self.escala, self.ox, self.oy = 1.0, 0, 0
        self.vacio = vacio
        self.al_dibujar = None
        self.bind('<Configure>', lambda e: self.redibujar())

    def poner(self, rgb):
        self.rgb = rgb
        self.redibujar()

    def redibujar(self):
        self.delete('all')
        W, H = self.winfo_width(), self.winfo_height()
        if W < 20 or H < 20:
            return
        if self.rgb is None:
            if self.vacio:
                self.create_text(W / 2, H / 2, text=self.vacio, fill=CL['fuerte'], justify='center')
            return
        img, f = ajustar(self.rgb, W - 4, H - 4)
        self._foto = a_photo(img)
        self.escala = f
        h, w = img.shape[:2]
        self.ox, self.oy = (W - w) // 2, (H - h) // 2
        self.create_image(self.ox, self.oy, anchor='nw', image=self._foto)
        if self.al_dibujar:
            self.al_dibujar(self)

    def a_canvas(self, x, y):
        return self.ox + x * self.escala, self.oy + y * self.escala

    def a_imagen(self, cx, cy):
        h, w = self.rgb.shape[:2]
        x = min(max((cx - self.ox) / self.escala, 0), w - 1)
        y = min(max((cy - self.oy) / self.escala, 0), h - 1)
        return x, y


def _nombre_seguro(texto):
    return re.sub(r'[^\w\-.]+', '_', texto).strip('_')


# ----------------------------------------------------------------------------
# aplicación
# ----------------------------------------------------------------------------
class AppSIFT:
    CLAVES = ['Médica', 'Telemétrica']

    def __init__(self, ruta_medica='Medica.jpg', ruta_tele='Telemetrica.jpg'):
        self.imgs, self.rois, self.rutas, self.aviso_carga = {}, {}, {}, {}
        self.cache = {}
        self.dt_ultimo = 0.0
        self.ultimo = None
        self._job = None
        self._tmp = None
        self._ancla = None
        for clave, ruta in zip(self.CLAVES, [ruta_medica, ruta_tele]):
            self._cargar(clave, ruta)
        self._cargar_rois()

    # ------------------------------------------------------------------ datos
    def _roi_defecto(self, rgb):
        H, W = rgb.shape[:2]
        return (int(W * 0.30), int(H * 0.30), int(W * 0.35), int(H * 0.35))

    def _cargar(self, clave, ruta):
        try:
            rgb = leer_imagen(ruta)
            aviso = f'Cargada «{ruta}» ({rgb.shape[1]}×{rgb.shape[0]} px)'
            real = True
        except Exception as e:
            rgb = imagen_demo(clave)
            aviso = (f'No se pudo abrir «{ruta}» ({type(e).__name__}). Se usa una imagen sintética '
                     'de demostración hasta que se cargue otra.')
            real = False
        self.imgs[clave] = rgb
        self.rutas[clave] = ruta
        self.rois[clave] = self._roi_defecto(rgb)
        self.aviso_carga[clave] = (aviso, real)
        self.cache = {k: v for k, v in self.cache.items() if k[1] != clave}

    def _guardar_rois(self):
        """Guarda las áreas elegidas para no repetir la selección al volver a ejecutar."""
        try:
            with open('rois.json', 'w') as f:
                json.dump({k: [int(v) for v in r] for k, r in self.rois.items()}, f)
        except OSError:
            pass

    def _cargar_rois(self):
        try:
            with open('rois.json') as f:
                datos = json.load(f)
        except (OSError, ValueError):
            return
        for k, v in datos.items():
            if k in self.imgs and len(v) == 4:
                H, W = self.imgs[k].shape[:2]
                x, y, w, h = v
                if x >= 0 and y >= 0 and w >= 16 and h >= 16 and x + w <= W and y + h <= H:
                    self.rois[k] = tuple(int(t) for t in v)

    def parametros(self):
        return dict(
            nfeatures=int(round(self.v_nf.get())), nOctaveLayers=int(round(self.v_capas.get())),
            contrastThreshold=round(self.v_contraste.get(), 4), edgeThreshold=round(self.v_borde.get(), 2),
            sigma=round(self.v_sigma.get(), 2), preproc=self.v_preproc.get(),
            ksize=int(round(self.v_ksize.get())), matcher=self.v_matcher.get(), ratio=self.v_ratio.get(),
            modelo=self.v_modelo.get(), ransac_thr=self.v_ransac.get(),
            min_inliers=int(round(self.v_inl.get())))

    def _extraer_cache(self, clave_cache, rgb, p):
        k = tuple(clave_cache) + claves_sift(p)
        if k not in self.cache:
            if len(self.cache) > 60:
                self.cache.pop(next(iter(self.cache)))
            t0 = time.time()
            kp, des = extraer(rgb, p)
            self.cache[k] = (kp, des, time.time() - t0)
        kp, des, dt = self.cache[k]
        self.dt_ultimo = dt
        return kp, des

    # ------------------------------------------------------------- ventana
    def ejecutar(self, bloquear=True):
        """Abre la ventana. Con bloquear=True la celda queda ocupada hasta que se cierra."""
        try:
            import ctypes
            ctypes.windll.shcore.SetProcessDpiAwareness(1)   # nitidez en Windows con escalado
        except Exception:
            pass
        self.root = tk.Tk()
        self._construir()
        self._sync_imagen()
        self.root.after(200, self._refrescar)
        self.root.lift()
        self.root.attributes('-topmost', True)
        self.root.after(400, lambda: self.root.attributes('-topmost', False))
        if bloquear:
            self.root.mainloop()

    def _estilos(self):
        st = ttk.Style(self.root)
        st.theme_use('clam')
        fuente = ('Segoe UI', 10)
        st.configure('.', background=CL['fondo'], foreground=CL['texto'], font=fuente)
        st.configure('TFrame', background=CL['fondo'])
        st.configure('TLabel', background=CL['fondo'], foreground=CL['texto'])
        st.configure('TLabelframe', background=CL['fondo'], bordercolor=CL['borde'])
        st.configure('TLabelframe.Label', background=CL['fondo'], foreground=CL['oscuro'],
                     font=('Segoe UI', 10, 'bold'))
        st.configure('TNotebook', background=CL['fondo'], borderwidth=0)
        st.configure('TNotebook.Tab', background=CL['suave'], foreground=CL['texto'], padding=(14, 6),
                     font=('Segoe UI', 10, 'bold'))
        st.map('TNotebook.Tab', background=[('selected', CL['fuerte'])], foreground=[('selected', 'white')])
        for nombre, color, activo, fg in (('P', CL['oscuro'], CL['fuerte'], 'white'),
                                          ('S', CL['medio'], CL['fuerte'], 'white'),
                                          ('L', CL['suave'], CL['borde'], CL['oscuro'])):
            st.configure(f'{nombre}.TButton', background=color, foreground=fg, padding=(8, 6),
                         borderwidth=0, font=('Segoe UI', 10, 'bold'))
            st.map(f'{nombre}.TButton', background=[('active', activo)])
        st.configure('Horizontal.TScale', background=CL['fuerte'], troughcolor=CL['suave'],
                     bordercolor=CL['borde'], lightcolor=CL['fuerte'], darkcolor=CL['fuerte'])
        st.map('Horizontal.TScale', background=[('active', CL['oscuro'])])
        for w in ('TRadiobutton', 'TCheckbutton'):
            st.configure(w, background=CL['fondo'], foreground=CL['texto'])
            st.map(w, background=[('active', CL['tarjeta'])])
        st.configure('TCombobox', fieldbackground='white', background=CL['suave'],
                     arrowcolor=CL['oscuro'], foreground=CL['texto'])
        st.map('TCombobox', fieldbackground=[('readonly', 'white')])
        st.configure('TEntry', fieldbackground='white')
        st.configure('TSpinbox', fieldbackground='white', arrowcolor=CL['oscuro'], background=CL['suave'])
        st.configure('Treeview', background='white', fieldbackground='white', foreground=CL['texto'],
                     rowheight=24)
        st.configure('Treeview.Heading', background=CL['fuerte'], foreground='white',
                     font=('Segoe UI', 10, 'bold'))
        self.root.option_add('*TCombobox*Listbox.background', 'white')
        self.root.option_add('*TCombobox*Listbox.selectBackground', CL['fuerte'])

    # ----- widgets auxiliares
    def _escala(self, padre, texto, var, desde, hasta, paso, comando, fila):
        """Deslizador con el valor al lado; se ajusta al paso al soltar y entonces avisa."""
        ttk.Label(padre, text=texto).grid(row=fila, column=0, sticky='w', padx=(6, 4), pady=3)
        entero = float(paso).is_integer()
        valor = ttk.Label(padre, width=6, anchor='e', foreground=CL['oscuro'],
                          font=('Segoe UI', 10, 'bold'))
        valor.grid(row=fila, column=2, padx=(4, 8))

        decimales = 0 if entero else (3 if paso < 0.01 else 2 if paso < 0.1 else 1)

        def formato(v):
            return f'{v:.{decimales}f}'

        def mover(v):
            valor.config(text=formato(float(v)))

        def soltar(_=None):
            v = round(round((var.get() - desde) / paso) * paso + desde, 6)
            var.set(min(max(v, desde), hasta))
            mover(var.get())
            comando()

        s = ttk.Scale(padre, variable=var, from_=desde, to=hasta, orient='horizontal', command=mover)
        s.grid(row=fila, column=1, sticky='ew', pady=3)
        s.bind('<ButtonRelease-1>', soltar)
        s.bind('<KeyRelease>', soltar)
        var.trace_add('write', lambda *a: mover(var.get()))
        mover(var.get())
        return s

    def _combo(self, padre, texto, var, valores, comando, fila, ancho=30):
        ttk.Label(padre, text=texto).grid(row=fila, column=0, sticky='w', padx=(6, 4), pady=4)
        c = ttk.Combobox(padre, textvariable=var, values=valores, state='readonly', width=ancho)
        c.grid(row=fila, column=1, sticky='ew', padx=(0, 6), pady=4)
        c.bind('<<ComboboxSelected>>', lambda e: comando())
        return c

    def _aviso(self, etiqueta, texto, tipo='info'):
        colores = {'ok': ('#E6DAFA', CL['oscuro']), 'warn': ('#F9DCEA', '#7A1745'),
                   'info': (CL['tarjeta'], CL['texto'])}
        bg, fg = colores[tipo]
        etiqueta.config(text=texto, bg=bg, fg=fg)

    def _panel_sift(self, padre):
        f = ttk.Frame(padre)
        f.columnconfigure(1, weight=1)
        self._combo(f, 'Configuración rápida', self.v_preset, list(PRESETS_SIFT), self._cambia_preset_sift, 0)
        ref = self.pedir_refresco
        self._escala(f, 'nfeatures', self.v_nf, 0, 5000, 100, ref, 1)
        self._escala(f, 'nOctaveLayers', self.v_capas, 1, 6, 1, ref, 2)
        self._escala(f, 'contrastThreshold', self.v_contraste, 0.005, 0.20, 0.005, ref, 3)
        self._escala(f, 'edgeThreshold', self.v_borde, 2.0, 40.0, 0.5, ref, 4)
        self._escala(f, 'sigma', self.v_sigma, 0.8, 3.2, 0.1, ref, 5)
        self._combo(f, 'Preprocesado', self.v_preproc,
                    ['Ninguno', 'Suavizado gaussiano', 'CLAHE', 'Suavizado + CLAHE'], ref, 6)
        self._escala(f, 'Tamaño del suavizado', self.v_ksize, 3, 15, 2, ref, 7)
        return f

    # ------------------------------------------------------------ construcción
    def _construir(self):
        r = self.root
        r.title('Práctica 4 · Detección de características con SIFT')
        r.geometry('1380x880')
        r.minsize(1150, 760)
        r.configure(bg=CL['fondo'])
        self._estilos()

        # variables compartidas (los controles repetidos en varias pestañas quedan sincronizados)
        self.sel_img = tk.StringVar(value='Médica')
        self.ruta_v = tk.StringVar()
        self.v_preset = tk.StringVar(value=list(PRESETS_SIFT)[0])
        self.v_nf, self.v_capas = tk.DoubleVar(value=0), tk.DoubleVar(value=3)
        self.v_contraste, self.v_borde = tk.DoubleVar(value=0.04), tk.DoubleVar(value=10.0)
        self.v_sigma, self.v_ksize = tk.DoubleVar(value=1.6), tk.DoubleVar(value=5)
        self.v_preproc = tk.StringVar(value='Ninguno')
        self.v_matcher = tk.StringVar(value='Fuerza bruta (BF)')
        self.v_ratio, self.v_modelo = tk.DoubleVar(value=0.75), tk.StringVar(value=MODELOS[0])
        self.v_ransac, self.v_inl = tk.DoubleVar(value=3.0), tk.DoubleVar(value=10)
        self.v_origen = tk.StringVar(value='Versión generada')
        self.v_archivo = tk.StringVar()
        self.v_vpreset = tk.StringVar(value='Rotación 30°')
        self.v_tx, self.v_ty = tk.DoubleVar(value=0), tk.DoubleVar(value=0)
        self.v_esc, self.v_ang = tk.DoubleVar(value=1.0), tk.DoubleVar(value=30.0)
        self.v_auto, self.v_gt = tk.BooleanVar(value=True), tk.BooleanVar(value=False)
        self.rx, self.ry = tk.IntVar(), tk.IntVar()
        self.rw, self.rh = tk.IntVar(), tk.IntVar()

        # cabecera con degradado lila
        cab = tk.Canvas(r, height=60, highlightthickness=0, bg=CL['oscuro'])
        cab.pack(fill='x')

        def pintar_cab(e=None):
            cab.delete('all')
            W = max(cab.winfo_width(), 1)
            for x in range(0, W, 4):
                cab.create_rectangle(x, 0, x + 4, 60, outline='',
                                     fill=mezclar(CL['oscuro'], CL['medio'], x / W))
            cab.create_text(18, 22, anchor='w', fill='white', font=('Segoe UI', 16, 'bold'),
                            text='Práctica 4 · Detección de características con SIFT')
            cab.create_text(18, 43, anchor='w', fill='#EADFFB', font=('Segoe UI', 9),
                            text='Procesamiento de Imágenes, Audio y Vídeo')
        cab.bind('<Configure>', pintar_cab)

        # barra de imagen
        barra = ttk.Frame(r, padding=(10, 8, 10, 0))
        barra.pack(fill='x')
        ttk.Radiobutton(barra, text='Imagen médica', value='Médica', variable=self.sel_img,
                        command=self._cambia_imagen).pack(side='left', padx=(0, 10))
        ttk.Radiobutton(barra, text='Imagen telemétrica', value='Telemétrica', variable=self.sel_img,
                        command=self._cambia_imagen).pack(side='left', padx=(0, 16))
        ttk.Label(barra, text='Ruta:').pack(side='left')
        ttk.Entry(barra, textvariable=self.ruta_v, width=55).pack(side='left', padx=6)
        ttk.Button(barra, text='Examinar…', style='L.TButton', command=self._examinar).pack(side='left', padx=2)
        ttk.Button(barra, text='Cargar', style='S.TButton', command=self._cargar_click).pack(side='left', padx=2)
        self.lbl_carga = tk.Label(r, anchor='w', justify='left', padx=10, pady=4, wraplength=1300)
        self.lbl_carga.pack(fill='x', padx=10, pady=(6, 2))

        self.nb = ttk.Notebook(r)
        self.nb.pack(fill='both', expand=True, padx=10, pady=(4, 10))
        t1, t2, t3 = (ttk.Frame(self.nb, padding=8) for _ in range(3))
        self.nb.add(t1, text='1 · Parámetros SIFT (a)')
        self.nb.add(t2, text='2 · Área de interés (b)')
        self.nb.add(t3, text='3 · Búsqueda (c)')
        self._pestana1(t1)
        self._pestana2(t2)
        self._pestana3(t3)
        self.nb.bind('<<NotebookTabChanged>>', lambda e: self.pedir_refresco())

    def _pestana1(self, t):
        t.columnconfigure(1, weight=1)
        t.rowconfigure(0, weight=1)
        izq = ttk.Frame(t, width=420)
        izq.grid(row=0, column=0, sticky='ns', padx=(0, 10))
        self._panel_sift(izq).pack(fill='x')
        self.lbl_stats = tk.Label(izq, anchor='w', justify='left', padx=10, pady=8, wraplength=370,
                                  bg=CL['tarjeta'], fg=CL['texto'])
        self.lbl_stats.pack(fill='x', padx=6, pady=(10, 4))
        self.v_hist = Visor(izq, vacio='Sin puntos', width=370, height=165)
        self.v_hist.pack(padx=6, pady=4)
        tk.Label(izq, anchor='w', justify='left', wraplength=370, padx=10, pady=8, bg=CL['tarjeta'],
                 fg=CL['texto'],
                 text=('contrastThreshold: bajarlo da más puntos, más débiles.\n'
                       'edgeThreshold: subirlo acepta más puntos sobre bordes.\n'
                       'nfeatures: conserva solo los N más fuertes (0 = todos).')
                 ).pack(fill='x', padx=6, pady=(6, 0))
        self.v_sift = Visor(t, vacio='Cargando…')
        self.v_sift.grid(row=0, column=1, sticky='nsew')

    def _pestana2(self, t):
        t.columnconfigure(1, weight=1)
        t.rowconfigure(0, weight=1)
        izq = ttk.Frame(t, width=360)
        izq.grid(row=0, column=0, sticky='ns', padx=(0, 10))
        tk.Label(izq, anchor='w', justify='left', wraplength=330, padx=10, pady=8, bg=CL['tarjeta'],
                 fg=CL['texto'],
                 text='El área se selecciona arrastrando el ratón sobre la imagen. '
                      'También se puede ajustar con los valores numéricos.').pack(fill='x', padx=6, pady=(0, 8))
        cuad = ttk.Frame(izq)
        cuad.pack(fill='x', padx=6)
        self.spins = []
        for i, (txt, var) in enumerate((('x', self.rx), ('y', self.ry), ('ancho', self.rw), ('alto', self.rh))):
            ttk.Label(cuad, text=txt).grid(row=i // 2, column=(i % 2) * 2, sticky='w', padx=(0, 4), pady=4)
            sp = ttk.Spinbox(cuad, textvariable=var, from_=0, to=1000, width=7, command=self._spin_cambia)
            sp.grid(row=i // 2, column=(i % 2) * 2 + 1, padx=(0, 14))
            sp.bind('<Return>', self._spin_cambia)
            sp.bind('<FocusOut>', self._spin_cambia)
            self.spins.append(sp)
        ttk.Button(izq, text='Área central por defecto', style='L.TButton',
                   command=self._reset_roi).pack(fill='x', padx=6, pady=8)
        self.lbl_roi = tk.Label(izq, anchor='w', justify='left', padx=10, pady=8, wraplength=330)
        self.lbl_roi.pack(fill='x', padx=6)
        ttk.Label(izq, text='Área seleccionada con sus puntos SIFT',
                  font=('Segoe UI', 10, 'bold'), foreground=CL['oscuro']).pack(anchor='w', padx=6, pady=(10, 2))
        self.v_crop = Visor(izq, vacio='', width=340, height=260)
        self.v_crop.pack(padx=6, pady=2)
        self.v_roi = Visor(t, vacio='Cargando…', cursor='crosshair')
        self.v_roi.grid(row=0, column=1, sticky='nsew')
        self.v_roi.al_dibujar = self._overlay_roi
        self.v_roi.bind('<ButtonPress-1>', self._m_press)
        self.v_roi.bind('<B1-Motion>', self._m_move)
        self.v_roi.bind('<ButtonRelease-1>', self._m_release)

    def _pestana3(self, t):
        t.columnconfigure(1, weight=1)
        t.rowconfigure(0, weight=1)
        izq = ttk.Frame(t, width=420)
        izq.grid(row=0, column=0, sticky='ns', padx=(0, 10))
        sub = ttk.Notebook(izq)
        sub.pack(fill='x')

        # --- versión de la imagen
        a = ttk.Frame(sub, padding=4)
        a.columnconfigure(1, weight=1)
        sub.add(a, text='Versión')
        f0 = ttk.Frame(a)
        f0.grid(row=0, column=0, columnspan=2, sticky='ew', pady=(2, 4))
        ttk.Radiobutton(f0, text='Versión generada', value='Versión generada', variable=self.v_origen,
                        command=self._pedir_busqueda).pack(side='left', padx=6)
        ttk.Radiobutton(f0, text='Archivo de imagen', value='Archivo de imagen', variable=self.v_origen,
                        command=self._pedir_busqueda).pack(side='left', padx=6)
        ttk.Label(a, text='Archivo').grid(row=1, column=0, sticky='w', padx=(6, 4))
        fa = ttk.Frame(a)
        fa.grid(row=1, column=1, sticky='ew', padx=(0, 6))
        fa.columnconfigure(0, weight=1)
        ttk.Entry(fa, textvariable=self.v_archivo).grid(row=0, column=0, sticky='ew')
        ttk.Button(fa, text='…', width=3, style='L.TButton', command=self._examinar_escena).grid(row=0, column=1, padx=(4, 0))
        self._combo(a, 'Versión predefinida', self.v_vpreset, list(VERSIONES) + ['Personalizada'],
                    self._cambia_vpreset, 2, ancho=26)
        mv = self._pedir_version
        self._escala(a, 'Traslación X (px)', self.v_tx, -400, 400, 5, mv, 3)
        self._escala(a, 'Traslación Y (px)', self.v_ty, -400, 400, 5, mv, 4)
        self._escala(a, 'Escala', self.v_esc, 0.25, 3.0, 0.05, mv, 5)
        self._escala(a, 'Rotación (º, antihoraria)', self.v_ang, -180, 180, 1, mv, 6)

        # --- parámetros SIFT (los mismos de la pestaña 1)
        b = ttk.Frame(sub, padding=4)
        sub.add(b, text='SIFT')
        self._panel_sift(b).pack(fill='x')

        # --- emparejamiento
        c = ttk.Frame(sub, padding=4)
        c.columnconfigure(1, weight=1)
        sub.add(c, text='Emparejamiento')
        pb = self._pedir_busqueda
        self._combo(c, 'Emparejador', self.v_matcher, ['Fuerza bruta (BF)', 'FLANN'], pb, 0, ancho=26)
        self._escala(c, 'Ratio test (Lowe)', self.v_ratio, 0.50, 0.95, 0.01, pb, 1)
        self._combo(c, 'Modelo RANSAC', self.v_modelo, MODELOS, pb, 2, ancho=26)
        self._escala(c, 'Umbral RANSAC (px)', self.v_ransac, 1.0, 10.0, 0.5, pb, 3)
        self._escala(c, 'Inliers mínimos', self.v_inl, 4, 60, 1, pb, 4)

        ttk.Checkbutton(izq, text='Buscar automáticamente al cambiar algo', variable=self.v_auto
                        ).pack(anchor='w', padx=6, pady=(10, 0))
        ttk.Checkbutton(izq, text='Mostrar la posición real (línea violeta fina)', variable=self.v_gt,
                        command=self._pedir_busqueda).pack(anchor='w', padx=6, pady=(2, 8))
        bot = ttk.Frame(izq)
        bot.pack(fill='x', padx=6)
        bot.columnconfigure((0, 1), weight=1)
        ttk.Button(bot, text='Buscar área de interés', style='P.TButton', command=self._buscar
                   ).grid(row=0, column=0, columnspan=2, sticky='ew', pady=2)
        ttk.Button(bot, text='Evaluar todas las versiones', style='S.TButton', command=self._evaluar_todas
                   ).grid(row=1, column=0, sticky='ew', padx=(0, 2), pady=2)
        ttk.Button(bot, text='Resumen visual', style='S.TButton', command=self._resumen_visual
                   ).grid(row=1, column=1, sticky='ew', padx=(2, 0), pady=2)
        ttk.Button(bot, text='Guardar versión', style='L.TButton', command=self._guardar_version
                   ).grid(row=2, column=0, sticky='ew', padx=(0, 2), pady=2)
        ttk.Button(bot, text='Guardar captura', style='L.TButton', command=self._guardar_captura
                   ).grid(row=2, column=1, sticky='ew', padx=(2, 0), pady=2)

        der = ttk.Frame(t)
        der.grid(row=0, column=1, sticky='nsew')
        der.columnconfigure(0, weight=1)
        der.rowconfigure(1, weight=3)
        der.rowconfigure(3, weight=2)
        self.lbl_estado = tk.Label(der, anchor='w', justify='left', padx=10, pady=8, wraplength=860,
                                   bg=CL['tarjeta'], fg=CL['texto'],
                                   text='Pendiente de ejecutar la búsqueda.')
        self.lbl_estado.grid(row=0, column=0, sticky='ew', pady=(0, 4))
        self.lbl_estado.bind('<Configure>', lambda e: self.lbl_estado.config(wraplength=max(200, e.width - 24)))
        self.v_res = Visor(der, vacio='Resultado de la búsqueda\n(recuadro rojo = área encontrada)')
        self.v_res.grid(row=1, column=0, sticky='nsew')
        ttk.Label(der, text='Emparejamientos (líneas lilas = inliers)',
                  font=('Segoe UI', 10, 'bold'), foreground=CL['oscuro']).grid(row=2, column=0, sticky='w', pady=(6, 2))
        self.v_emp = Visor(der, vacio='')
        self.v_emp.grid(row=3, column=0, sticky='nsew')

    # --------------------------------------------------------- eventos generales
    def pedir_refresco(self):
        """Agrupa varios cambios seguidos en una sola actualización."""
        if self._job:
            self.root.after_cancel(self._job)
        self._job = self.root.after(120, self._refrescar)

    def _refrescar(self):
        self._job = None
        i = self.nb.index(self.nb.select())
        self.root.config(cursor='watch')
        self.root.update_idletasks()
        try:
            if i == 0:
                self._vista_sift()
            elif i == 1:
                self._vista_roi()
            elif i == 2 and self.v_auto.get():
                self._buscar()
        finally:
            self.root.config(cursor='')

    def _pedir_busqueda(self, *_):
        if self.nb.index(self.nb.select()) == 2 and self.v_auto.get():
            self.pedir_refresco()

    def _pedir_version(self):
        self.v_vpreset.set('Personalizada')
        self.v_origen.set('Versión generada')
        self._pedir_busqueda()

    def _cambia_vpreset(self):
        nombre = self.v_vpreset.get()
        if nombre in VERSIONES:
            v = VERSIONES[nombre]
            self.v_tx.set(v['tx'])
            self.v_ty.set(v['ty'])
            self.v_esc.set(v['escala'])
            self.v_ang.set(v['angulo'])
        self.v_origen.set('Versión generada')
        self._pedir_busqueda()

    def _cambia_preset_sift(self):
        pr = PRESETS_SIFT[self.v_preset.get()]
        self.v_nf.set(pr['nfeatures'])
        self.v_capas.set(pr['capas'])
        self.v_contraste.set(pr['contraste'])
        self.v_borde.set(pr['borde'])
        self.v_sigma.set(pr['sigma'])
        self.pedir_refresco()

    def _sync_imagen(self):
        clave = self.sel_img.get()
        self.ruta_v.set(self.rutas[clave])
        aviso, real = self.aviso_carga[clave]
        self._aviso(self.lbl_carga, aviso, 'ok' if real else 'warn')
        H, W = self.imgs[clave].shape[:2]
        for sp, mn, mx in zip(self.spins, (0, 0, 16, 16), (W - 16, H - 16, W, H)):
            sp.config(from_=mn, to=mx)
        self._sync_roi_vars()

    def _sync_roi_vars(self):
        x, y, w, h = self.rois[self.sel_img.get()]
        self.rx.set(x)
        self.ry.set(y)
        self.rw.set(w)
        self.rh.set(h)

    def _cambia_imagen(self):
        self._sync_imagen()
        self.v_res.poner(None)
        self.v_emp.poner(None)
        self._aviso(self.lbl_estado, 'Pendiente de ejecutar la búsqueda.', 'info')
        self.pedir_refresco()

    def _cargar_click(self):
        self._cargar(self.sel_img.get(), self.ruta_v.get().strip())
        self._cambia_imagen()

    def _examinar(self):
        ruta = filedialog.askopenfilename(title='Elegir imagen', parent=self.root,
                                          filetypes=[('Imágenes', '*.jpg *.jpeg *.png *.bmp *.tif *.tiff'),
                                                     ('Todos', '*.*')])
        if ruta:
            self.ruta_v.set(ruta)
            self._cargar_click()

    def _examinar_escena(self):
        ruta = filedialog.askopenfilename(title='Elegir la imagen alterada', parent=self.root,
                                          filetypes=[('Imágenes', '*.jpg *.jpeg *.png *.bmp *.tif *.tiff'),
                                                     ('Todos', '*.*')])
        if ruta:
            self.v_archivo.set(ruta)
            self.v_origen.set('Archivo de imagen')
            self._pedir_busqueda()

    # ------------------------------------------------------------- área de interés
    def _spin_cambia(self, _=None):
        clave = self.sel_img.get()
        H, W = self.imgs[clave].shape[:2]
        try:
            x, y, w, h = self.rx.get(), self.ry.get(), self.rw.get(), self.rh.get()
        except tk.TclError:
            self._sync_roi_vars()
            return
        x = max(0, min(x, W - 16))
        y = max(0, min(y, H - 16))
        w = max(16, min(w, W - x))
        h = max(16, min(h, H - y))
        self.rois[clave] = (x, y, w, h)
        self._sync_roi_vars()
        self._guardar_rois()
        self._vista_roi()

    def _reset_roi(self):
        clave = self.sel_img.get()
        self.rois[clave] = self._roi_defecto(self.imgs[clave])
        self._sync_roi_vars()
        self._guardar_rois()
        self._vista_roi()

    def _m_press(self, e):
        if self.v_roi.rgb is not None:
            self._ancla = self.v_roi.a_imagen(e.x, e.y)

    def _m_move(self, e):
        if self._ancla is None:
            return
        x1, y1 = self.v_roi.a_imagen(e.x, e.y)
        x0, y0 = self._ancla
        self._tmp = (int(min(x0, x1)), int(min(y0, y1)), int(abs(x1 - x0)), int(abs(y1 - y0)))
        self._overlay_roi(self.v_roi)

    def _m_release(self, e):
        tmp, self._ancla, self._tmp = self._tmp, None, None
        if tmp and tmp[2] >= 16 and tmp[3] >= 16:
            self.rois[self.sel_img.get()] = tmp
            self._sync_roi_vars()
            self._guardar_rois()
        self._vista_roi()

    def _overlay_roi(self, v):
        v.delete('roi')
        if v.rgb is None:
            return
        x, y, w, h = self._tmp or self.rois[self.sel_img.get()]
        x0, y0 = v.a_canvas(x, y)
        x1, y1 = v.a_canvas(x + w, y + h)
        v.create_rectangle(x0, y0, x1, y1, outline='#FF0000', width=2, tags='roi')

    def _plantilla(self):
        clave = self.sel_img.get()
        x, y, w, h = self.rois[clave]
        return self.imgs[clave][y:y + h, x:x + w].copy()

    # ------------------------------------------------------------------ vistas
    def _vista_sift(self):
        clave = self.sel_img.get()
        rgb, p = self.imgs[clave], self.parametros()
        kp, des = self._extraer_cache(('img', clave), rgb, p)
        vis = cv2.drawKeypoints(rgb, kp, None, color=CL['punto'],
                                flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
        self.v_sift.poner(vis)
        tam = np.array([k.size for k in kp])
        texto = (f'Puntos clave: {len(kp)}\n'
                 f'Tamaño mediano: {np.median(tam):.1f} px\n' if len(kp) else 'Puntos clave: 0\n')
        texto += f'Descriptores: {None if des is None else des.shape}\nExtracción: {self.dt_ultimo:.2f} s'
        self.lbl_stats.config(text=texto)
        self.v_hist.poner(histograma(tam) if len(kp) else None)

    def _vista_roi(self):
        clave = self.sel_img.get()
        rgb, p = self.imgs[clave], self.parametros()
        x, y, w, h = self.rois[clave]
        plantilla = self._plantilla()
        kp, _ = self._extraer_cache(('roi', clave, x, y, w, h), plantilla, p)
        self.v_roi.poner(rgb)
        self.v_crop.poner(cv2.drawKeypoints(plantilla, kp, None, color=CL['punto'],
                                            flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS))
        if len(kp) < 10:
            self._aviso(self.lbl_roi, f'Área: x={x}, y={y}, {w}×{h} px\nSolo {len(kp)} puntos SIFT: es difícil '
                        'localizarla. Conviene elegir una zona con más estructura o bajar el umbral de contraste.',
                        'warn')
        else:
            self._aviso(self.lbl_roi, f'Área: x={x}, y={y}, {w}×{h} px\n{len(kp)} puntos SIFT', 'ok')

    # ---------------------------------------------------------------- búsqueda
    def _obtener_escena(self, clave):
        """Devuelve (escena, matriz 3x3 o None, clave de caché), o un texto de error en lugar de la escena."""
        base = self.imgs[clave]
        if self.v_origen.get() == 'Versión generada':
            v = (int(round(self.v_tx.get())), int(round(self.v_ty.get())), round(self.v_esc.get(), 3),
                 round(self.v_ang.get(), 1))
            escena, M = transformar(base, *v)
            return escena, M, ('esc', clave) + v
        ruta = self.v_archivo.get().strip()
        if not ruta:
            return 'Falta indicar la ruta de la imagen alterada.', None, None
        try:
            escena = leer_imagen(ruta)
        except Exception as e:
            return f'No se pudo abrir «{ruta}» ({type(e).__name__}).', None, None
        return escena, None, ('arch', clave, ruta, os.path.getmtime(ruta))

    def _ejecutar_busqueda(self, clave, p, escena, M, clave_esc):
        """Una búsqueda completa; devuelve el resultado, la plantilla y la posición real (si se conoce)."""
        base = self.imgs[clave]
        x, y, w, h = self.rois[clave]
        plantilla = base[y:y + h, x:x + w]
        kp_t, des_t = self._extraer_cache(('roi', clave, x, y, w, h), plantilla, p)
        kp_e, des_e = self._extraer_cache(clave_esc, escena, p)
        t0 = time.time()
        res = localizar(kp_t, des_t, kp_e, des_e, (h, w), p)
        res['tiempo'] = time.time() - t0
        res['kp_t'], res['kp_e'] = kp_t, kp_e
        quad_real = None
        res['iou'] = res['err'] = None
        if M is not None:
            esq = np.float32([[x, y], [x + w, y], [x + w, y + h], [x, y + h]]).reshape(-1, 1, 2)
            quad_real = cv2.perspectiveTransform(esq, M).reshape(-1, 2)
            if res['encontrado']:
                res['iou'] = iou_cuadrilateros(res['quad'], quad_real, escena.shape)
                res['err'] = float(np.mean(np.linalg.norm(res['quad'] - quad_real, axis=1)))
        return res, plantilla, quad_real

    def _buscar(self):
        clave = self.sel_img.get()
        p = self.parametros()
        escena, M, ce = self._obtener_escena(clave)
        if isinstance(escena, str):
            self._aviso(self.lbl_estado, escena, 'warn')
            return
        self.root.config(cursor='watch')
        self.root.update_idletasks()
        try:
            res, plantilla, quad_real = self._ejecutar_busqueda(clave, p, escena, M, ce)
        finally:
            self.root.config(cursor='')

        if res['encontrado']:
            txt = (f"Área de interés encontrada: {res['n_inliers']} inliers de {res['n_buenos']} emparejamientos "
                   f"({res['n_kp_t']} puntos en el área, {res['n_kp_e']} en la escena).")
            if res['iou'] is not None:
                txt += f"\nIoU con la posición real: {res['iou']:.3f}  ·  error medio de esquinas: {res['err']:.1f} px"
            self._aviso(self.lbl_estado, txt, 'ok')
        else:
            self._aviso(self.lbl_estado, 'No se ha encontrado el área de interés.\n' + res['motivo'], 'warn')

        grosor = max(2, round(max(escena.shape[:2]) / 220))
        final = dibujar_busqueda(escena, res, grosor, quad_real if self.v_gt.get() else None)
        emp = dibujar_emparejamientos(plantilla, res['kp_t'], escena, res['kp_e'], res)
        self.v_res.poner(final)
        self.v_emp.poner(emp)
        self.ultimo = (final, emp, f"{clave}_{self.v_vpreset.get()}")

    def _tabla_versiones(self, clave, p):
        base = self.imgs[clave]
        filas = []
        for nombre, v in VERSIONES.items():
            escena, M = transformar(base, v['tx'], v['ty'], v['escala'], v['angulo'])
            ce = ('esc', clave, v['tx'], v['ty'], round(v['escala'], 3), v['angulo'])
            res, _, _ = self._ejecutar_busqueda(clave, p, escena, M, ce)
            filas.append((nombre, escena, res))
        return filas

    def _evaluar_todas(self):
        clave = self.sel_img.get()
        self.root.config(cursor='watch')
        self.root.update_idletasks()
        try:
            filas = self._tabla_versiones(clave, self.parametros())
        finally:
            self.root.config(cursor='')
        n_ok = sum(r['encontrado'] for _, _, r in filas)
        win = tk.Toplevel(self.root)
        win.title(f'Evaluación de las versiones · imagen {clave.lower()}')
        win.configure(bg=CL['fondo'])
        win.geometry('900x400')
        tk.Label(win, bg=CL['tarjeta'], fg=CL['texto'], anchor='w', padx=10, pady=8,
                 text=f'Imagen {clave.lower()}: {n_ok} de {len(filas)} versiones encontradas con los parámetros actuales'
                 ).pack(fill='x', padx=10, pady=10)
        cols = ('version', 'enc', 'emp', 'inl', 'iou', 'err')
        tabla = ttk.Treeview(win, columns=cols, show='headings', height=len(filas))
        for c, t, a in zip(cols, ('Versión', '¿Encontrada?', 'Emparejamientos', 'Inliers', 'IoU', 'Error esquinas (px)'),
                           (330, 100, 120, 70, 70, 140)):
            tabla.heading(c, text=t)
            tabla.column(c, width=a, anchor='w' if c == 'version' else 'center')
        tabla.tag_configure('no', background='#F9DCEA', foreground='#7A1745')
        for nombre, _, r in filas:
            tabla.insert('', 'end', tags=() if r['encontrado'] else ('no',), values=(
                nombre, 'sí' if r['encontrado'] else 'NO', r['n_buenos'], r['n_inliers'],
                '—' if r['iou'] is None else f"{r['iou']:.3f}",
                '—' if r['err'] is None else f"{r['err']:.1f}"))
        tabla.pack(fill='both', expand=True, padx=10, pady=(0, 10))

    def _resumen_visual(self):
        clave = self.sel_img.get()
        self.root.config(cursor='watch')
        self.root.update_idletasks()
        try:
            filas = self._tabla_versiones(clave, self.parametros())
            columnas = 5
            nfil = math.ceil(len(filas) / columnas)
            fig = Figure(figsize=(3.6 * columnas, 3.4 * nfil + 0.6), dpi=100, facecolor=CL['fondo'])
            ejes = fig.subplots(nfil, columnas, squeeze=False).ravel()
            n_ok = 0
            for ax, (nombre, escena, res) in zip(ejes, filas):
                grosor = max(2, round(max(escena.shape[:2]) / 160))
                ax.imshow(dibujar_busqueda(escena, res, grosor))
                if res['encontrado']:
                    n_ok += 1
                    estado, color = f"IoU {res['iou']:.2f} · {res['n_inliers']} inliers", CL['oscuro']
                else:
                    estado, color = 'NO encontrada', CL['aviso']
                ax.set_title(f"{nombre.split(' (')[0]}\n{estado}", fontsize=9, color=color, fontweight='bold')
                ax.axis('off')
            for ax in ejes[len(filas):]:
                ax.axis('off')
            fig.suptitle(f'Imagen {clave.lower()}: {n_ok} de {len(filas)} versiones encontradas',
                         color=CL['oscuro'], fontsize=13, fontweight='bold')
            fig.tight_layout()
            os.makedirs('resultados', exist_ok=True)
            ruta = f'resultados/resumen_{_nombre_seguro(clave.lower())}.png'
            fig.savefig(ruta, dpi=130, facecolor=fig.get_facecolor())
            imagen = figura_a_rgb(fig)
        finally:
            self.root.config(cursor='')
        win = tk.Toplevel(self.root)
        win.title(f'Resumen visual · imagen {clave.lower()}')
        win.configure(bg=CL['fondo'])
        win.geometry('1300x640')
        v = Visor(win)
        v.pack(fill='both', expand=True, padx=8, pady=8)
        v.poner(imagen)
        self._aviso(self.lbl_estado, f'Resumen visual guardado en «{ruta}»', 'ok')

    # ---------------------------------------------------------------- guardado
    def _guardar_version(self):
        clave = self.sel_img.get()
        if self.v_origen.get() != 'Versión generada':
            self._aviso(self.lbl_estado, 'Solo se pueden guardar las versiones generadas por la aplicación.', 'warn')
            return
        escena, _, _ = self._obtener_escena(clave)
        os.makedirs('versiones', exist_ok=True)
        ruta = f"versiones/{_nombre_seguro(clave.lower() + '_' + self.v_vpreset.get())}.png"
        ok, buf = cv2.imencode('.png', cv2.cvtColor(escena, cv2.COLOR_RGB2BGR))
        if ok:
            buf.tofile(ruta)
            self._aviso(self.lbl_estado, f'Versión guardada en «{ruta}»', 'ok')

    def _guardar_captura(self):
        if self.ultimo is None:
            self._aviso(self.lbl_estado, 'Todavía no hay ningún resultado que guardar.', 'warn')
            return
        final, emp, nombre = self.ultimo
        ancho = 1400
        piezas = [cv2.resize(im, (ancho, int(im.shape[0] * ancho / im.shape[1])), interpolation=cv2.INTER_AREA)
                  for im in (final, emp)]
        captura = np.vstack(piezas)
        os.makedirs('resultados', exist_ok=True)
        ruta = f'resultados/busqueda_{_nombre_seguro(nombre)}.png'
        ok, buf = cv2.imencode('.png', cv2.cvtColor(captura, cv2.COLOR_RGB2BGR))
        if ok:
            buf.tofile(ruta)
            self._aviso(self.lbl_estado, f'Captura guardada en «{ruta}»', 'ok')

## 5. Ejecución

La ventana se abre fuera del notebook. Mientras permanece abierta, la celda sigue en ejecución, y termina cuando se cierra la ventana.

In [23]:
app = AppSIFT(ruta_medica='Medica.jpg', ruta_tele='Telemetrica.jpg')
app.ejecutar()

## 6. Análisis de lo que permite comprobar la aplicación

**Apartado a.** El parámetro que más cambia el resultado es `contrastThreshold`: al bajarlo aparecen muchos más puntos, pero la mayoría son débiles y poco discriminantes; al subirlo quedan solo los más fuertes. `edgeThreshold` controla cuántos puntos situados sobre bordes se aceptan, y `nfeatures` limita el total a los más fuertes. En las resonancias, con poco contraste, la configuración "más sensible" da más puntos que la de OpenCV por defecto. El histograma de tamaños muestra en qué escalas se concentran.

**Apartado b.** Un área de interés con poca textura (fondo liso, zonas homogéneas) tiene pocos puntos SIFT y es mucho más difícil de localizar. Por eso la interfaz muestra el número de puntos del área y avisa cuando son pocos.

**Apartado c.** SIFT es invariante a la traslación, a la rotación y, dentro de un rango, a la escala, así que cabe esperar que las versiones predefinidas se localicen sin tocar parámetros. La tabla de evaluación y el resumen visual permiten comprobarlo versión por versión con el IoU y el error de esquinas. Donde falla, los casos más probables son los siguientes:
- **Escalas muy pequeñas.** El área pierde detalle y hay menos puntos que superen el ratio test. Ayuda bajar `contrastThreshold`, activar CLAHE o subir algo el ratio.
- **Zonas repetitivas.** En las imágenes de satélite (edificios, árboles) y en las resonancias (cerebro casi simétrico) hay emparejamientos ambiguos. Un ratio más estricto o más inliers mínimos reducen los falsos positivos.
- **Umbral RANSAC demasiado alto.** Acepta emparejamientos incorrectos como inliers.

Quedan fuera de esta entrega el apartado d (deformaciones de la imagen) y el apartado opcional de corregir la imagen alterada. La opción de homografía ya está disponible en la aplicación como base para ambos.